In [ ]:
# 本文件主要进行截面回归
%run config_0.ipynb
%run data_io_1.ipynb
%run clean_and_universe_2.ipynb
%run features_3.ipynb
%run monthly_4.ipynb
%run portfolio_5.ipynb
%run time_series_regression_6.ipynb

import pandas as pd
import numpy as np
import statsmodels.api as sm

In [ ]:
# 截面回归的第一步为时间序列回归，回归之后得到暴露矩阵B，然后对R（月度超额收益率）取平均得到E_R,E_R的维度为1*N，因此最后回归估计得到1*K的E_F

# 获得E_R，但直接列操作得到的是N*1的矩阵，因此等号右侧也需要根据矩阵的转置法则转换
# 若需要1*N的矩阵需要一下操作：
"""
E_R = E_R.reset_index('E_R').reset_columns(stock_code)
"""
def get_E_R(monthly):
    E_R = monthly.groupby(stock_code)[monthly_excess_ret].mean().rename('E_R')
    return E_R

In [ ]:
# 进行截面回归：E_R (N,) 对 B^T (N x K) 回归，得到因子风险溢价 E_F (K,)
#
# 为什么用 B.T 而不是 B：
#   时序回归的设定是 R(T×N) = F(T×K) @ B(K×N) + α
#   —— B 的每一列是「某只股票的 K 个因子暴露」，所以 B 是 K×N（因子 × 股票）。
#   截面回归把角色反过来：
#     被解释变量 = N 只股票的（平均）超额收益 E_R (N×1)；
#     解释变量   = 这 N 只股票各自的因子暴露；
#     待估系数   = 因子风险溢价 λ (K×1)。
#   截面方程（每只股票一行）：E_R[n] = α + Σ_k λ_k · B[k, n] + ε，
#   写成矩阵就是 E_R(N×1) = B.T(N×K) @ λ(K×1) + α，
#   因此设计矩阵必须「行 = 股票、列 = 因子」，也就是 B 的转置 B.T。
def cross_section_regression(E_R , B):
    X = sm.add_constant(B.T).reindex(E_R.index)   # N x (K+1)，并按股票对齐
    valid = E_R.notna() & X.notna().all(axis = 1)
    res = sm.OLS(E_R[valid] , X[valid]).fit(cov_type = 'HC1')   # White 异方差稳健标准误
    E_F = res.params.drop('const')
    E_F_t = res.tvalues.drop('const')            # 风险溢价的 t 值
    alpha = res.params['const']
    return E_F , E_F_t , alpha , res


df = read_daily_data()
df = cleaning(df)
df = universe(df)
cf_df = read_cf_data()
cf_df = cf_cleaning(cf_df)
bs_df = read_bs_data()
bs_df = bs_cleaning(bs_df)
df = get_market_feature(df)
df = get_momentum(df)   # 与 monthly_4 / portfolio_5 的内联流程一致：动量列必须先构造
df = get_cp(df , cf_df)
df = get_asset_growth(df , bs_df)
df = get_excess_return(df)
df = get_bm(df)
df = get_ep(df)
df = get_one_month_volatility(df)
df = get_max_return(df)
df = get_illiquidity(df)
df = get_turnover_12m(df)
df = get_abn_turnover_1m(df)
monthly = get_ym_monthly_excessret(df)
snap = get_form_signal(df)
monthly_final = get_monthly_panel(monthly , snap)
F = get_groups_neutral(monthly_final)
R , T , N= get_R(monthly)
B , _ , _ = time_series_regression(R , F)
E_R = get_E_R(monthly)
E_F , E_F_t , alpha , res = cross_section_regression(E_R , B)
print('查看截面回归的风险溢价 E_F 及其 t 值')
print(E_F)
print(E_F_t)
print('截距 alpha:', alpha)